# 04 — Normalizar autores y afiliaciones UNAM

Este notebook aplica únicamente decisiones ya revisadas y congeladas en:

- `Diccionario_autore.csv`
- `Diccionario_afiliaciones.csv`
- `Afiliaciones_contextuales.csv`
- `Registros_externos_UNAM.csv`

El archivo de control utilizado para construir estas decisiones es
`../00_control/UNAM_Completo_Corregido.csv`.

Reglas definitivas de esta versión:

- las 131 filas de `Registros_externos_UNAM.csv` están aprobadas como `EXCLUIR`;
- una fila conservada debe terminar con una o dos afiliaciones UNAM;
- las afiliaciones externas desaparecen de las filas conservadas;
- CECAv se estandariza únicamente como `Centro de Estudios en Computación Avanzada`;
- `Aplicar` se respeta explícitamente en ambos diccionarios;
- solo pueden cambiar `Autor_norm`, `Afiliacion1` y `Afiliacion2`;
- las otras 12 columnas se validan como inmutables;
- la salida final conserva exactamente 15 columnas.


In [1]:
import os
from collections import Counter
import unicodedata
import pandas as pd

# ============================================================
# 04 - NORMALIZAR AUTORES Y AFILIACIONES UNAM
# Aplicación determinista de decisiones ya revisadas.
# ============================================================

archivo_entrada = "../04_Limpieza/01_Internos_unam/autores_unam_separados.csv"

carpeta_salida = "../04_Limpieza/02_normalizacion"

archivo_dic_autores = f"{carpeta_salida}/Diccionario_autore.csv"
archivo_dic_afiliaciones = f"{carpeta_salida}/Diccionario_afiliaciones.csv"
archivo_contextuales = f"{carpeta_salida}/Afiliaciones_contextuales.csv"
archivo_externos = f"{carpeta_salida}/Registros_externos_UNAM.csv"

archivo_salida = f"{carpeta_salida}/autores_unam_normalizados.csv"
archivo_excluidos = f"{carpeta_salida}/registros_excluidos_no_UNAM.csv"
archivo_auditoria = f"{carpeta_salida}/auditoria_normalizacion_autores_afiliaciones.csv"

os.makedirs(carpeta_salida, exist_ok=True)

columnas_canonicas = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2",
    "ISBN",
    "ISSN",
    "Doi",
    "URL",
    "Area",
    "SubArea",
    "Keywords",
    "Abstract"
]

columnas_modificables = [
    "Autor_norm",
    "Afiliacion1",
    "Afiliacion2"
]

columnas_inmutables = [
    c for c in columnas_canonicas
    if c not in columnas_modificables
]

clave_base = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_norm",
    "Doi",
    "Afiliacion1",
    "Afiliacion2"
]

clave_contexto = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_actual",
    "Doi",
    "Afiliacion1_original",
    "Afiliacion2_original"
]

esquema_dic_autores = [
    "Autor_actual",
    "Autor_final",
    "Origen_nombre",
    "Aplicar",
    "Comentario"
]

esquema_dic_afiliaciones = [
    "Afiliacion_actual",
    "Afiliacion1_final",
    "Afiliacion2_final",
    "Clasificacion",
    "Origen",
    "Aplicar",
    "Comentario"
]

esquema_contextuales = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_actual",
    "Doi",
    "Afiliacion1_original",
    "Afiliacion2_original",
    "Afiliacion1_final",
    "Afiliacion2_final",
    "Metodo_resolucion",
    "Evidencia",
    "Candidata_exclusion_no_UNAM"
]

esquema_externos = [
    "Fuente_origen",
    "indice",
    "Titulo",
    "Año",
    "Autor_actual",
    "Autor_canonico",
    "Afiliacion1_original",
    "Afiliacion2_original",
    "Doi",
    "Evidencia",
    "Motivo",
    "Decision"
]

clasificaciones_afiliacion = {
    "UNAM",
    "UNAM_DOBLE",
    "GENERICA_UNAM",
    "EXTERNA",
    "AMBIGUA"
}

aplicar_autor = {"SI", "NO"}
aplicar_afiliacion = {"SI", "NO", "REVISAR"}

cecav_canonica = "Centro de Estudios en Computación Avanzada"


# ============================================================
# FUNCIONES
# ============================================================

def leer_csv(ruta):
    return pd.read_csv(
        ruta,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig"
    )


def validar_columnas(df, columnas, nombre):
    if list(df.columns) != columnas:
        raise ValueError(
            f"{nombre}: estructura inesperada.\n"
            f"Esperadas: {columnas}\n"
            f"Encontradas: {list(df.columns)}"
        )


def es_nfc(texto):
    return unicodedata.normalize("NFC", texto) == texto


def crear_claves(df, columnas):
    return pd.Series(
        list(df[columnas].itertuples(index=False, name=None)),
        index=df.index,
        dtype="object"
    )


def valores_finales_afiliacion(fila):
    valores = []

    for columna in ["Afiliacion1_final", "Afiliacion2_final"]:
        valor = fila[columna].strip()

        if valor and valor not in valores:
            valores.append(valor)

    return valores


def agregar_sin_repetir(destino, valores):
    for valor in valores:
        if valor and valor not in destino:
            destino.append(valor)


def validar_cecav(valor, origen):
    if not valor:
        return

    texto = valor.lower()

    if (
        "centro de estudios" in texto
        and ("computacion avanzada" in texto or "computación avanzada" in texto)
        and valor != cecav_canonica
    ):
        raise ValueError(
            f"{origen}: quedó una forma no canónica de CECAv: {valor!r}"
        )


# ============================================================
# 1. CARGA
# ============================================================

entrada = leer_csv(archivo_entrada)
dic_autores = leer_csv(archivo_dic_autores)
dic_afiliaciones = leer_csv(archivo_dic_afiliaciones)
contextuales = leer_csv(archivo_contextuales)
externos = leer_csv(archivo_externos)

validar_columnas(entrada, columnas_canonicas, "autores_unam_separados.csv")
validar_columnas(dic_autores, esquema_dic_autores, "Diccionario_autore.csv")
validar_columnas(
    dic_afiliaciones,
    esquema_dic_afiliaciones,
    "Diccionario_afiliaciones.csv"
)
validar_columnas(
    contextuales,
    esquema_contextuales,
    "Afiliaciones_contextuales.csv"
)
validar_columnas(
    externos,
    esquema_externos,
    "Registros_externos_UNAM.csv"
)

entrada_original = entrada.copy(deep=True)

if entrada.shape != (5445, 15):
    raise ValueError(
        f"La entrada esperada para estos diccionarios es (5445, 15) "
        f"y se encontró {entrada.shape}."
    )

print("=== CARGA ===")
print("Filas de entrada:", len(entrada))
print("Columnas:", len(entrada.columns))
print("Variantes de autor:", len(dic_autores))
print("Afiliaciones del diccionario:", len(dic_afiliaciones))
print("Resoluciones contextuales:", len(contextuales))
print("Exclusiones definitivas:", len(externos))
print()


# ============================================================
# 2. VALIDACIONES DE LA ENTRADA
# ============================================================

if entrada["Autor_norm"].str.strip().eq("").any():
    raise ValueError("La entrada contiene Autor_norm vacío.")

if entrada["Afiliacion1"].str.strip().eq("").any():
    raise ValueError("La entrada contiene Afiliacion1 vacío.")

for columna in columnas_modificables:
    if not entrada[columna].map(es_nfc).all():
        raise ValueError(
            f"La entrada contiene valores no NFC en {columna}."
        )


# ============================================================
# 3. VALIDAR EXCLUSIONES DEFINITIVAS
# ============================================================

if externos["Decision"].str.strip().eq("").any():
    raise ValueError(
        "Registros_externos_UNAM.csv contiene decisiones vacías."
    )

if set(externos["Decision"]) != {"EXCLUIR"}:
    raise ValueError(
        "Todos los registros de Registros_externos_UNAM.csv "
        "deben estar aprobados explícitamente como EXCLUIR."
    )

claves_base = crear_claves(entrada, clave_base)
conteo_claves_base = Counter(claves_base.tolist())

claves_externos = crear_claves(externos, clave_contexto)

if claves_externos.duplicated().any():
    raise ValueError(
        "Registros_externos_UNAM.csv contiene llaves duplicadas."
    )

faltan_externos = [
    clave
    for clave in claves_externos
    if conteo_claves_base.get(clave, 0) == 0
]

ambiguos_externos = [
    clave
    for clave in claves_externos
    if conteo_claves_base.get(clave, 0) > 1
]

if faltan_externos:
    raise ValueError(
        f"{len(faltan_externos)} exclusiones no existen en la base actual."
    )

if ambiguos_externos:
    raise ValueError(
        f"{len(ambiguos_externos)} exclusiones coinciden con más de una fila."
    )

set_externos = set(claves_externos.tolist())


# ============================================================
# 4. VALIDAR DICCIONARIO DE AUTORES
# ============================================================

if dic_autores["Autor_actual"].duplicated().any():
    raise ValueError(
        "Diccionario_autore.csv contiene Autor_actual duplicado."
    )

if dic_autores["Autor_final"].str.strip().eq("").any():
    raise ValueError(
        "Diccionario_autore.csv contiene Autor_final vacío."
    )

estados_autor = set(dic_autores["Aplicar"])

if not estados_autor.issubset(aplicar_autor):
    raise ValueError(
        f"Estados no válidos en Aplicar de autores: "
        f"{sorted(estados_autor - aplicar_autor)}"
    )

autores_base = set(entrada["Autor_norm"])
autores_dic = set(dic_autores["Autor_actual"])

if autores_base != autores_dic:
    raise ValueError(
        "El diccionario de autores no corresponde exactamente "
        "a las variantes de la entrada."
    )

if not dic_autores["Autor_final"].map(es_nfc).all():
    raise ValueError(
        "Diccionario_autore.csv contiene Autor_final no NFC."
    )

con_guion = dic_autores["Autor_final"].str.contains(
    r"[-‐‑‒–—−]",
    regex=True
)

con_coma = dic_autores["Autor_final"].str.contains(
    ",",
    regex=False
)

if con_guion.any() or con_coma.any():
    raise ValueError(
        "Se encontraron guiones o comas en Autor_final."
    )

con_punto = dic_autores["Autor_final"].str.contains(
    ".",
    regex=False
)

puntos_no_manuales = (
    con_punto
    & dic_autores["Origen_nombre"].ne("Revision_manual")
)

if puntos_no_manuales.any():
    raise ValueError(
        "Hay puntos en Autor_final fuera de decisiones Revision_manual."
    )

excepciones_punto = dic_autores.loc[
    con_punto,
    ["Autor_actual", "Autor_final", "Comentario"]
].copy()


# ============================================================
# 5. VALIDAR DICCIONARIO DE AFILIACIONES
# ============================================================

if dic_afiliaciones["Afiliacion_actual"].duplicated().any():
    raise ValueError(
        "Diccionario_afiliaciones.csv contiene Afiliacion_actual duplicada."
    )

if set(dic_afiliaciones["Clasificacion"]) - clasificaciones_afiliacion:
    raise ValueError(
        "Diccionario_afiliaciones.csv contiene clasificaciones no válidas."
    )

if set(dic_afiliaciones["Aplicar"]) - aplicar_afiliacion:
    raise ValueError(
        "Diccionario_afiliaciones.csv contiene estados Aplicar no válidos."
    )

afiliaciones_base = set(
    pd.concat(
        [entrada["Afiliacion1"], entrada["Afiliacion2"]],
        ignore_index=True
    )
)

afiliaciones_base.discard("")

afiliaciones_dic = set(dic_afiliaciones["Afiliacion_actual"])

if afiliaciones_base != afiliaciones_dic:
    raise ValueError(
        "El diccionario de afiliaciones no corresponde exactamente "
        "a las afiliaciones de la entrada."
    )

for _, fila in dic_afiliaciones.iterrows():

    finales = valores_finales_afiliacion(fila)

    for valor in finales:
        validar_cecav(valor, "Diccionario_afiliaciones.csv")

    if len(finales) > 2:
        raise ValueError(
            f"Una afiliación produce más de dos valores finales: "
            f"{fila['Afiliacion_actual']!r}"
        )

    if fila["Aplicar"] == "NO":
        if (
            fila["Afiliacion1_final"] != fila["Afiliacion_actual"]
            or fila["Afiliacion2_final"] != ""
        ):
            raise ValueError(
                "Aplicar=NO significa conservar exactamente la afiliación actual. "
                f"Revisar: {fila['Afiliacion_actual']!r}"
            )

    if fila["Clasificacion"] == "EXTERNA":
        if finales:
            raise ValueError(
                f"Una afiliación EXTERNA produce salida UNAM: "
                f"{fila['Afiliacion_actual']!r}"
            )

    elif fila["Aplicar"] in {"SI", "NO"} and not finales:
        raise ValueError(
            f"Una afiliación UNAM resuelta quedó sin salida final: "
            f"{fila['Afiliacion_actual']!r}"
        )

    if (
        fila["Clasificacion"] == "UNAM_DOBLE"
        and fila["Aplicar"] in {"SI", "NO"}
        and len(finales) != 2
    ):
        raise ValueError(
            f"UNAM_DOBLE debe producir dos afiliaciones: "
            f"{fila['Afiliacion_actual']!r}"
        )

for columna in ["Afiliacion1_final", "Afiliacion2_final"]:
    valores = dic_afiliaciones.loc[
        dic_afiliaciones[columna].str.strip().ne(""),
        columna
    ]

    if not valores.map(es_nfc).all():
        raise ValueError(
            f"Valores no NFC en {columna} del diccionario de afiliaciones."
        )


# ============================================================
# 6. VALIDAR RESOLUCIONES CONTEXTUALES
# ============================================================

claves_contexto = crear_claves(
    contextuales,
    clave_contexto
)

if claves_contexto.duplicated().any():
    raise ValueError(
        "Afiliaciones_contextuales.csv contiene llaves duplicadas."
    )

for _, fila in contextuales.iterrows():

    finales = []

    agregar_sin_repetir(
        finales,
        [
            fila["Afiliacion1_final"].strip(),
            fila["Afiliacion2_final"].strip()
        ]
    )

    if not finales:
        raise ValueError(
            "Existe una resolución contextual sin afiliación final."
        )

    if len(finales) > 2:
        raise ValueError(
            "Existe una resolución contextual con más de dos afiliaciones."
        )

    for valor in finales:
        validar_cecav(
            valor,
            "Afiliaciones_contextuales.csv"
        )

faltan_contextos = [
    clave
    for clave in claves_contexto
    if conteo_claves_base.get(clave, 0) == 0
]

ambiguos_contextos = [
    clave
    for clave in claves_contexto
    if conteo_claves_base.get(clave, 0) > 1
]

if faltan_contextos:
    raise ValueError(
        f"{len(faltan_contextos)} resoluciones contextuales "
        "no existen en la base."
    )

if ambiguos_contextos:
    raise ValueError(
        f"{len(ambiguos_contextos)} resoluciones contextuales "
        "coinciden con más de una fila."
    )

set_contextos = set(claves_contexto.tolist())

if set_externos & set_contextos:
    raise ValueError(
        "Una fila aparece a la vez en exclusiones y resoluciones contextuales."
    )

print("Validaciones previas: OK")
print()


# ============================================================
# 7. EXCLUIR REGISTROS DEFINITIVAMENTE EXTERNOS
# ============================================================

mascara_externa = claves_base.map(
    lambda clave: clave in set_externos
)

if int(mascara_externa.sum()) != len(externos):
    raise ValueError(
        "La cantidad de exclusiones no coincide con "
        "Registros_externos_UNAM.csv."
    )

registros_excluidos = entrada.loc[
    mascara_externa,
    columnas_canonicas
].copy()

trabajo = entrada.loc[
    ~mascara_externa,
    columnas_canonicas
].copy()

print("=== EXCLUSIÓN DEFINITIVA ===")
print("Filas de entrada:", len(entrada))
print("Registros externos eliminados:", len(registros_excluidos))
print("Filas que continúan:", len(trabajo))
print()


# ============================================================
# 8. NORMALIZAR AUTORES
# ============================================================

dic_autores_idx = dic_autores.set_index("Autor_actual")

autores_finales = []

for autor in trabajo["Autor_norm"]:

    decision = dic_autores_idx.loc[autor]

    if decision["Aplicar"] == "SI":
        autor_final = decision["Autor_final"]

    elif decision["Aplicar"] == "NO":
        if decision["Autor_final"] != autor:
            raise ValueError(
                "Un autor conservado tiene Aplicar=NO pero "
                "Autor_final es distinto de Autor_actual: "
                f"{autor!r}"
            )

        autor_final = autor

    else:
        raise ValueError(
            f"Estado Aplicar no reconocido para {autor!r}"
        )

    autores_finales.append(autor_final)

trabajo["Autor_norm"] = autores_finales

if trabajo["Autor_norm"].str.strip().eq("").any():
    raise ValueError(
        "La normalización generó autores vacíos."
    )


# ============================================================
# 9. PREPARAR RESOLUCIONES DE AFILIACIONES
# ============================================================

dic_afiliaciones_idx = dic_afiliaciones.set_index(
    "Afiliacion_actual"
)

mapa_contextual = {}

for _, fila in contextuales.iterrows():

    clave = tuple(
        fila[c]
        for c in clave_contexto
    )

    finales = []

    agregar_sin_repetir(
        finales,
        [
            fila["Afiliacion1_final"].strip(),
            fila["Afiliacion2_final"].strip()
        ]
    )

    mapa_contextual[clave] = (
        finales[0],
        finales[1] if len(finales) == 2 else ""
    )

valores_revisar = set(
    dic_afiliaciones.loc[
        dic_afiliaciones["Aplicar"].eq("REVISAR"),
        "Afiliacion_actual"
    ]
)

filas_revisar_sin_contexto = []

for idx, fila in entrada.loc[trabajo.index].iterrows():

    clave = claves_base.loc[idx]

    if clave in set_contextos:
        continue

    afiliaciones = {
        fila["Afiliacion1"],
        fila["Afiliacion2"]
    }

    afiliaciones.discard("")

    conflictivas = afiliaciones & valores_revisar

    if conflictivas:
        filas_revisar_sin_contexto.append(
            (idx, sorted(conflictivas))
        )

if filas_revisar_sin_contexto:
    raise ValueError(
        "Existen afiliaciones REVISAR sin resolución contextual. "
        f"Primeros casos: {filas_revisar_sin_contexto[:10]}"
    )


# ============================================================
# 10. NORMALIZAR AFILIACIONES
# ============================================================

afiliacion1_final = []
afiliacion2_final = []
metodos_afiliacion = []

for idx, fila in entrada.loc[trabajo.index].iterrows():

    clave = claves_base.loc[idx]

    if clave in mapa_contextual:

        f1, f2 = mapa_contextual[clave]

        finales = []

        agregar_sin_repetir(
            finales,
            [f1, f2]
        )

        metodo = "CONTEXTUAL"

    else:

        finales = []

        for campo in ["Afiliacion1", "Afiliacion2"]:

            afiliacion = fila[campo]

            if afiliacion == "":
                continue

            decision = dic_afiliaciones_idx.loc[afiliacion]

            if decision["Aplicar"] == "REVISAR":
                raise ValueError(
                    "Se intentó aplicar globalmente una afiliación REVISAR: "
                    f"{afiliacion!r}"
                )

            if decision["Aplicar"] == "NO":
                valores = [afiliacion]

            else:
                valores = valores_finales_afiliacion(decision)

            agregar_sin_repetir(
                finales,
                valores
            )

        metodo = "GLOBAL"

    if len(finales) > 2:
        raise ValueError(
            "Una fila produciría más de dos afiliaciones UNAM.\n"
            f"Índice pandas: {idx}\n"
            f"Valores: {finales}"
        )

    if not finales:
        raise ValueError(
            "Una fila conservada quedó sin afiliación UNAM.\n"
            f"Índice pandas: {idx}\n"
            f"Fuente: {fila['Fuente_origen']}\n"
            f"indice: {fila['indice']}\n"
            f"Autor: {fila['Autor_norm']}"
        )

    for valor in finales:
        validar_cecav(
            valor,
            "Salida normalizada"
        )

    afiliacion1_final.append(finales[0])
    afiliacion2_final.append(
        finales[1] if len(finales) == 2 else ""
    )
    metodos_afiliacion.append(metodo)

trabajo["Afiliacion1"] = afiliacion1_final
trabajo["Afiliacion2"] = afiliacion2_final


# ============================================================
# 11. VALIDACIONES FINALES
# ============================================================

if list(trabajo.columns) != columnas_canonicas:
    raise ValueError(
        "La salida no conserva exactamente las 15 columnas canónicas."
    )

filas_esperadas = len(entrada) - len(externos)

if len(trabajo) != filas_esperadas:
    raise ValueError(
        f"Se esperaban {filas_esperadas} filas y se obtuvieron "
        f"{len(trabajo)}."
    )

entrada_conservada = entrada_original.loc[trabajo.index]

for columna in columnas_inmutables:

    if not entrada_conservada[columna].equals(
        trabajo[columna]
    ):
        diferencias = (
            entrada_conservada[columna]
            != trabajo[columna]
        ).sum()

        raise ValueError(
            f"Se modificó una columna prohibida: {columna}. "
            f"Diferencias: {diferencias}"
        )

if trabajo["Autor_norm"].str.strip().eq("").any():
    raise ValueError(
        "La salida contiene Autor_norm vacío."
    )

if trabajo["Afiliacion1"].str.strip().eq("").any():
    raise ValueError(
        "La salida contiene Afiliacion1 vacío."
    )

duplicadas = (
    trabajo["Afiliacion2"].str.strip().ne("")
    & trabajo["Afiliacion1"].eq(
        trabajo["Afiliacion2"]
    )
)

if duplicadas.any():
    raise ValueError(
        f"Hay {int(duplicadas.sum())} filas con "
        "Afiliacion1 == Afiliacion2."
    )

for columna in columnas_modificables:

    if not trabajo[columna].map(es_nfc).all():
        raise ValueError(
            f"La salida contiene valores no NFC en {columna}."
        )

for columna in ["Afiliacion1", "Afiliacion2"]:
    for valor in trabajo[columna].unique():
        validar_cecav(
            valor,
            f"Salida/{columna}"
        )

claves_salida_originales = claves_base.loc[trabajo.index]

if claves_salida_originales.map(
    lambda clave: clave in set_externos
).any():
    raise ValueError(
        "Sobrevivió un registro aprobado para exclusión."
    )

print("Validaciones finales: OK")
print()


# ============================================================
# 12. AUDITORÍA
# ============================================================

autor_cambios = (
    entrada_conservada["Autor_norm"].reset_index(drop=True)
    != trabajo["Autor_norm"].reset_index(drop=True)
).sum()

af1_cambios = (
    entrada_conservada["Afiliacion1"].reset_index(drop=True)
    != trabajo["Afiliacion1"].reset_index(drop=True)
).sum()

af2_cambios = (
    entrada_conservada["Afiliacion2"].reset_index(drop=True)
    != trabajo["Afiliacion2"].reset_index(drop=True)
).sum()

metodos = Counter(
    metodos_afiliacion
)

afiliaciones_unicas = (
    set(trabajo["Afiliacion1"])
    | set(trabajo["Afiliacion2"])
) - {""}

formas_cecav = sorted(
    afiliacion
    for afiliacion in afiliaciones_unicas
    if (
        "centro de estudios" in afiliacion.lower()
        and (
            "computacion avanzada" in afiliacion.lower()
            or "computación avanzada" in afiliacion.lower()
        )
    )
)

if formas_cecav != [cecav_canonica]:
    raise ValueError(
        f"CECAv no quedó totalmente estandarizado: {formas_cecav}"
    )

auditoria = pd.DataFrame(
    [
        ("Filas_entrada", len(entrada)),
        ("Columnas_entrada", len(entrada.columns)),
        ("Variantes_autor_diccionario", len(dic_autores)),
        ("Variantes_afiliacion_diccionario", len(dic_afiliaciones)),
        ("Resoluciones_contextuales", len(contextuales)),
        ("Registros_externos_aprobados", len(externos)),
        ("Registros_externos_eliminados", len(registros_excluidos)),
        ("Filas_salida", len(trabajo)),
        ("Columnas_salida", len(trabajo.columns)),
        ("Filas_autor_modificado", int(autor_cambios)),
        ("Filas_Afiliacion1_modificada", int(af1_cambios)),
        ("Filas_Afiliacion2_modificada", int(af2_cambios)),
        ("Filas_afiliacion_contextual", metodos["CONTEXTUAL"]),
        ("Filas_afiliacion_global", metodos["GLOBAL"]),
        (
            "Filas_con_dos_afiliaciones_UNAM",
            int(trabajo["Afiliacion2"].str.strip().ne("").sum())
        ),
        ("Autores_unicos_antes", entrada["Autor_norm"].nunique()),
        ("Autores_unicos_salida", trabajo["Autor_norm"].nunique()),
        (
            "Afiliaciones_UNAM_unicas_salida",
            len(afiliaciones_unicas)
        ),
        (
            "Forma_CECAv_final",
            cecav_canonica
        ),
        (
            "Excepciones_manual_con_punto_en_Autor_final",
            len(excepciones_punto)
        ),
        ("Errores_integridad", 0)
    ],
    columns=["Metrica", "Valor"]
)


# ============================================================
# 13. GUARDAR
# ============================================================

salida = trabajo[
    columnas_canonicas
].reset_index(drop=True)

excluidos_salida = registros_excluidos[
    columnas_canonicas
].reset_index(drop=True)

salida.to_csv(
    archivo_salida,
    index=False,
    encoding="utf-8-sig"
)

excluidos_salida.to_csv(
    archivo_excluidos,
    index=False,
    encoding="utf-8-sig"
)

auditoria.to_csv(
    archivo_auditoria,
    index=False,
    encoding="utf-8-sig"
)

verificacion = leer_csv(
    archivo_salida
)

validar_columnas(
    verificacion,
    columnas_canonicas,
    "autores_unam_normalizados.csv"
)

if verificacion.shape != salida.shape:
    raise ValueError(
        "El archivo guardado no tiene las dimensiones esperadas."
    )

if not verificacion.equals(salida):
    raise ValueError(
        "El archivo guardado no coincide exactamente con la salida en memoria."
    )


# ============================================================
# 14. RESUMEN
# ============================================================

print("=== RESULTADO ===")
print("Archivo final:", archivo_salida)
print("Registros excluidos:", archivo_excluidos)
print("Auditoría:", archivo_auditoria)
print()
print("Filas entrada:", len(entrada))
print("Externos eliminados:", len(registros_excluidos))
print("Filas salida:", len(salida))
print("Autores modificados:", int(autor_cambios))
print("Afiliacion1 modificada:", int(af1_cambios))
print("Afiliacion2 modificada:", int(af2_cambios))
print("Resoluciones contextuales:", metodos["CONTEXTUAL"])
print("Normalizaciones globales:", metodos["GLOBAL"])
print(
    "Filas con dos afiliaciones UNAM:",
    int(salida["Afiliacion2"].str.strip().ne("").sum())
)
print("Autores únicos finales:", salida["Autor_norm"].nunique())
print("Afiliaciones UNAM únicas finales:", len(afiliaciones_unicas))
print("CECAv:", formas_cecav[0])
print()
print("Proceso terminado sin errores de integridad.")


=== CARGA ===
Filas de entrada: 5445
Columnas: 15
Variantes de autor: 1358
Afiliaciones del diccionario: 986
Resoluciones contextuales: 663
Exclusiones definitivas: 131

Validaciones previas: OK

=== EXCLUSIÓN DEFINITIVA ===
Filas de entrada: 5445
Registros externos eliminados: 131
Filas que continúan: 5314

Validaciones finales: OK

=== RESULTADO ===
Archivo final: ../04_Limpieza/02_normalizacion/autores_unam_normalizados.csv
Registros excluidos: ../04_Limpieza/02_normalizacion/registros_excluidos_no_UNAM.csv
Auditoría: ../04_Limpieza/02_normalizacion/auditoria_normalizacion_autores_afiliaciones.csv

Filas entrada: 5445
Externos eliminados: 131
Filas salida: 5314
Autores modificados: 5139
Afiliacion1 modificada: 5245
Afiliacion2 modificada: 821
Resoluciones contextuales: 663
Normalizaciones globales: 4651
Filas con dos afiliaciones UNAM: 331
Autores únicos finales: 690
Afiliaciones UNAM únicas finales: 71
CECAv: Centro de Estudios en Computación Avanzada

Proceso terminado sin errores